# Triggers y Procedimientos

CREATE TRIGGER en SQLite, procedimientos almacenados en PostgreSQL y MySQL

## Introducción

- Los triggers y las vistas son herramientas avanzadas de SQL que permiten automatizar acciones en la base de datos y simplificar consultas complejas. Un trigger es un bloque de código que se ejecuta automáticamente ante eventos como INSERT, UPDATE o DELETE. Las vistas son consultas guardadas que actúan como tablas virtuales. En SQLite, dado que no existen procedimientos almacenados nativos, Python cumple ese rol mediante funciones que encapsulan lógica SQL reutilizable.
Objetivos de Aprendizaje

## Comprender qué es un trigger y cuándo se activa (BEFORE / AFTER)
- Escribir triggers para auditoría, validación y actualizaciones automáticas
- Crear vistas simples y complejas con CREATE VIEW
- Distinguir entre vistas actualizables y vistas de solo lectura
- Implementar el patrón de "procedimientos almacenados" usando funciones Python con sqlite3

## Qué es un Trigger

> Un trigger (disparador) es una rutina que la base de datos ejecuta automáticamente cuando ocurre un evento específico sobre una tabla: INSERT, UPDATE o DELETE. SQLite soporta los modificadores BEFORE y AFTER para controlar si la acción del trigger ocurre antes o después del evento. Dentro del trigger se pueden referenciar los valores nuevos con NEW y los valores anteriores con OLD.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Crear tabla de productos
cursor.execute("""
CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER DEFAULT 0
)
""")

# Trigger AFTER INSERT: imprime un aviso cuando se agrega un producto
cursor.execute("""
CREATE TRIGGER aviso_nuevo_producto
AFTER INSERT ON productos
BEGIN
    SELECT RAISE(IGNORE);
END
""")

# Insertar un producto
cursor.execute("INSERT INTO productos (nombre, precio, stock) VALUES ('Laptop', 999.99, 10)")
conn.commit()

# Verificar que el producto existe
cursor.execute("SELECT * FROM productos")
row = cursor.fetchone()
print(f"Producto insertado: id=$${row[0]}, nombre=$${row[1]}, precio=$${row[2]}, stock=$${row[3]}")

conn.close()
print("Trigger AFTER INSERT ejecutado correctamente.")


## Sintaxis CREATE TRIGGER en SQLite

> La sintaxis de CREATE TRIGGER en SQLite sigue el patrón: CREATE TRIGGER nombre BEFORE|AFTER INSERT|UPDATE|DELETE ON tabla BEGIN instrucciones; END. Los triggers BEFORE pueden cancelar la operación con RAISE(ABORT, 'mensaje'). Los triggers AFTER se usan para acciones secundarias como registros de auditoría. UPDATE puede limitarse a columnas específicas con OF columna.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tabla principal
cursor.execute("""
CREATE TABLE empleados (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    salario REAL NOT NULL
)
""")

# Tabla de auditoría
cursor.execute("""
CREATE TABLE auditoria_salarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    empleado_id INTEGER,
    salario_anterior REAL,
    salario_nuevo REAL,
    fecha TEXT DEFAULT (datetime('now'))
)
""")

# Trigger BEFORE UPDATE: valida que el salario no sea negativo
cursor.execute("""
CREATE TRIGGER validar_salario
BEFORE UPDATE OF salario ON empleados
BEGIN
    SELECT CASE
        WHEN NEW.salario < 0
        THEN RAISE(ABORT, 'El salario no puede ser negativo')
    END;
END
""")

# Trigger AFTER UPDATE: registra el cambio en la tabla de auditoría
cursor.execute("""
CREATE TRIGGER registrar_cambio_salario
AFTER UPDATE OF salario ON empleados
BEGIN
    INSERT INTO auditoria_salarios (empleado_id, salario_anterior, salario_nuevo)
    VALUES (OLD.id, OLD.salario, NEW.salario);
END
""")

cursor.execute("INSERT INTO empleados (nombre, salario) VALUES ('Ana García', 3000.0)")
conn.commit()

cursor.execute("UPDATE empleados SET salario = 3500.0 WHERE nombre = 'Ana García'")
conn.commit()

cursor.execute("SELECT * FROM auditoria_salarios")
registro = cursor.fetchone()
print(f"Auditoría: empleado_id=${registro[1]}, anterior=${registro[2]}, nuevo=${registro[3]}")

# Intentar salario negativo (debe fallar)
try:
    cursor.execute("UPDATE empleados SET salario = -100 WHERE nombre = 'Ana García'")
except (sqlite3.OperationalError, sqlite3.IntegrityError) as e:
    print(f"Error esperado: ${e}")

conn.close()


## Triggers para Auditoría y Auto-Updates

> Los casos de uso más comunes para triggers son: (1) Logs de auditoría que registran quién cambió qué y cuándo. (2) Actualizaciones automáticas como recalcular totales cuando se modifica un detalle. (3) Validación de datos que rechaza inserciones o actualizaciones inválidas antes de que lleguen a la tabla. En SQLite, NEW contiene los valores a insertar/actualizar y OLD los valores previos al cambio.


In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tablas de un carrito de compras
cursor.execute("""
CREATE TABLE ordenes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    cliente TEXT NOT NULL,
    total REAL DEFAULT 0.0
)
""")

cursor.execute("""
CREATE TABLE items_orden (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    orden_id INTEGER REFERENCES ordenes(id),
    producto TEXT NOT NULL,
    cantidad INTEGER NOT NULL,
    precio_unitario REAL NOT NULL
)
""")

# Trigger: recalcular total de la orden al agregar un item
cursor.execute("""
CREATE TRIGGER actualizar_total_insert
AFTER INSERT ON items_orden
BEGIN
    UPDATE ordenes
    SET total = (
        SELECT COALESCE(SUM(cantidad * precio_unitario), 0)
        FROM items_orden WHERE orden_id = NEW.orden_id
    )
    WHERE id = NEW.orden_id;
END
""")

# Trigger: recalcular total al eliminar un item
cursor.execute("""
CREATE TRIGGER actualizar_total_delete
AFTER DELETE ON items_orden
BEGIN
    UPDATE ordenes
    SET total = (
        SELECT COALESCE(SUM(cantidad * precio_unitario), 0)
        FROM items_orden WHERE orden_id = OLD.orden_id
    )
    WHERE id = OLD.orden_id;
END
""")

cursor.execute("INSERT INTO ordenes (cliente) VALUES ('Carlos López')")
orden_id = cursor.lastrowid

cursor.execute("INSERT INTO items_orden (orden_id, producto, cantidad, precio_unitario) VALUES (?, 'Mouse', 2, 25.0)", (orden_id,))
cursor.execute("INSERT INTO items_orden (orden_id, producto, cantidad, precio_unitario) VALUES (?, 'Teclado', 1, 75.0)", (orden_id,))
conn.commit()

cursor.execute("SELECT total FROM ordenes WHERE id = ?", (orden_id,))
total = cursor.fetchone()[0]
print(f"Total calculado automáticamente: ${total}")  # Esperado: 125.0

# Eliminar un item y verificar actualización automática
cursor.execute("DELETE FROM items_orden WHERE producto = 'Mouse'")
conn.commit()
cursor.execute("SELECT total FROM ordenes WHERE id = ?", (orden_id,))
total = cursor.fetchone()[0]
print(f"Total tras eliminar Mouse: ${total}")  # Esperado: 75.0

conn.close()


## CREATE VIEW — Vistas Simples y Complejas

> Una vista es una consulta SELECT guardada en la base de datos con un nombre, que se puede usar como si fuera una tabla. Las vistas simplifican consultas frecuentes, ocultan complejidad de JOINs, y pueden restringir qué columnas son visibles. En SQLite se crean con CREATE VIEW nombre AS SELECT .... Las vistas no almacenan datos; cada vez que se consultan, ejecutan su SELECT subyacente.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Esquema de base de datos de una empresa
cursor.execute("""
CREATE TABLE departamentos (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    presupuesto REAL DEFAULT 0
)
""")

cursor.execute("""
CREATE TABLE empleados (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    puesto TEXT,
    salario REAL,
    depto_id INTEGER REFERENCES departamentos(id)
)
""")

# Datos de ejemplo
cursor.executemany("INSERT INTO departamentos VALUES (?,?,?)", [
    (1, 'Ingeniería', 500000),
    (2, 'Marketing', 200000),
    (3, 'Finanzas', 300000)
])
cursor.executemany("INSERT INTO empleados (nombre, puesto, salario, depto_id) VALUES (?,?,?,?)", [
    ('Ana Torres', 'Dev Senior', 6000, 1),
    ('Luis Mora', 'Dev Junior', 3500, 1),
    ('Sara Ruiz', 'Diseñadora', 4000, 2),
    ('Pedro Díaz', 'Analista', 4500, 3),
])
conn.commit()

# Vista simple: empleados con nombre de departamento
cursor.execute("""
CREATE VIEW vista_empleados AS
SELECT e.id, e.nombre, e.puesto, e.salario, d.nombre AS departamento
FROM empleados e
JOIN departamentos d ON e.depto_id = d.id
""")

# Vista compleja: resumen por departamento
cursor.execute("""
CREATE VIEW resumen_departamentos AS
SELECT d.nombre AS departamento,
       COUNT(e.id) AS num_empleados,
       ROUND(AVG(e.salario), 2) AS salario_promedio,
       SUM(e.salario) AS masa_salarial
FROM departamentos d
LEFT JOIN empleados e ON e.depto_id = d.id
GROUP BY d.id, d.nombre
""")

print("=== Vista: Empleados con Departamento ===")
for row in cursor.execute("SELECT * FROM vista_empleados"):
    print(f"  $${row[1]} | $${row[2]} | $${row[3]} | Depto: $${row[4]}")

print("\n=== Vista: Resumen por Departamento ===")
for row in cursor.execute("SELECT * FROM resumen_departamentos"):
    print(f"  $${row[0]}: $${row[1]} empleados, promedio $${row[2]}, masa $${row[3]}")

conn.close()


## Vistas Actualizables vs Solo Lectura

> En SQLite, una vista es actualizable si se basa en una sola tabla sin GROUP BY, DISTINCT, funciones de agregación ni subconsultas. Para hacer una vista "escribible" se usan INSTEAD OF triggers que interceptan INSERT/UPDATE/DELETE sobre la vista y los redirigen a las tablas reales. Las vistas con JOINs o agregaciones son solo lectura a menos que se implementen estos triggers.


In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    email TEXT UNIQUE,
    activo INTEGER DEFAULT 1
)
""")

cursor.execute("""
CREATE TABLE perfiles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    usuario_id INTEGER REFERENCES usuarios(id),
    bio TEXT,
    avatar_url TEXT
)
""")

# Vista que une usuarios y perfiles (solo lectura sin INSTEAD OF trigger)
cursor.execute("""
CREATE VIEW vista_perfiles_completos AS
SELECT u.id, u.nombre, u.email, u.activo, p.bio, p.avatar_url
FROM usuarios u
LEFT JOIN perfiles p ON p.usuario_id = u.id
""")

# INSTEAD OF INSERT: permite insertar en la vista como si fuera una tabla
cursor.execute("""
CREATE TRIGGER insertar_perfil_completo
INSTEAD OF INSERT ON vista_perfiles_completos
BEGIN
    INSERT OR IGNORE INTO usuarios (nombre, email, activo)
    VALUES (NEW.nombre, NEW.email, COALESCE(NEW.activo, 1));

    INSERT INTO perfiles (usuario_id, bio, avatar_url)
    VALUES (
        (SELECT id FROM usuarios WHERE email = NEW.email),
        NEW.bio,
        NEW.avatar_url
    );
END
""")

# Insertar a través de la vista
cursor.execute("""
INSERT INTO vista_perfiles_completos (nombre, email, activo, bio, avatar_url)
VALUES ('María Solano', 'maria@ejemplo.com', 1, 'Desarrolladora Full Stack', 'https://img.ejemplo.com/maria.png')
""")
conn.commit()

print("=== Perfil insertado via vista ===")
for row in cursor.execute("SELECT * FROM vista_perfiles_completos"):
    print(f"  id=$${row[0]}, nombre=$${row[1]}, email=$${row[2]}")
    print(f"  bio=$${row[4]}, avatar=$${row[5]}")

# Verificar que se insertó en las tablas reales
cursor.execute("SELECT COUNT(*) FROM usuarios")
u = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM perfiles")
p = cursor.fetchone()[0]
print(f"\nRegistros reales: ${u} usuario(s), ${p} perfil(es)")

conn.close()


## Procedimientos con Funciones Python

> SQLite no tiene procedimientos almacenados nativos (stored procedures). En Python, el patrón equivalente es encapsular la lógica SQL en funciones reutilizables que reciben parámetros, ejecutan múltiples sentencias dentro de una transacción y manejan errores. Esto da los mismos beneficios: reutilización, encapsulamiento, mantenimiento centralizado y manejo de transacciones.


In [ ]:
import sqlite3

def crear_esquema(conn):
    """Crea las tablas necesarias."""
    c = conn.cursor()
    c.execute("""
    CREATE TABLE cuentas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        titular TEXT NOT NULL,
        saldo REAL NOT NULL DEFAULT 0 CHECK(saldo >= 0)
    )
    """)
    c.execute("""
    CREATE TABLE movimientos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        cuenta_id INTEGER REFERENCES cuentas(id),
        tipo TEXT CHECK(tipo IN ('deposito','retiro','transferencia')),
        monto REAL,
        fecha TEXT DEFAULT (datetime('now'))
    )
    """)
    conn.commit()

def transferir(conn, origen_id, destino_id, monto):
    """Stored procedure: transferencia entre cuentas."""
    if monto <= 0:
        raise ValueError("El monto debe ser positivo")
    c = conn.cursor()
    try:
        c.execute("UPDATE cuentas SET saldo = saldo - ? WHERE id = ?", (monto, origen_id))
        c.execute("UPDATE cuentas SET saldo = saldo + ? WHERE id = ?", (monto, destino_id))
        c.execute("INSERT INTO movimientos (cuenta_id, tipo, monto) VALUES (?, 'transferencia', ?)", (origen_id, monto))
        c.execute("INSERT INTO movimientos (cuenta_id, tipo, monto) VALUES (?, 'transferencia', ?)", (destino_id, monto))
        conn.commit()
        print(f"Transferencia de ${monto} de cuenta ${origen_id} a cuenta ${destino_id}: OK")
    except sqlite3.IntegrityError:
        conn.rollback()
        print("Error: saldo insuficiente. Operación revertida.")

conn = sqlite3.connect(':memory:')
crear_esquema(conn)
c = conn.cursor()
c.execute("INSERT INTO cuentas (titular, saldo) VALUES ('Cuenta A', 1000.0)")
c.execute("INSERT INTO cuentas (titular, saldo) VALUES ('Cuenta B', 500.0)")
conn.commit()

transferir(conn, 1, 2, 300.0)
transferir(conn, 1, 2, 9000.0)  # Debe fallar

for row in c.execute("SELECT titular, saldo FROM cuentas"):
    print(f"  $${row[0]}: $${row[1]}")
conn.close()


## Sistema de Auditoría Completo con Triggers

> Implementa un sistema de auditoría que registra automáticamente todos los cambios (inserciones, actualizaciones y eliminaciones) en una tabla de productos usando tres triggers separados.


In [ ]:
import sqlite3
from datetime import datetime

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tabla de productos
cursor.execute("""
CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER DEFAULT 0
)
""")

# Tabla de auditoría general
cursor.execute("""
CREATE TABLE log_auditoria (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tabla TEXT,
    operacion TEXT,
    registro_id INTEGER,
    detalle TEXT,
    fecha TEXT DEFAULT (datetime('now'))
)
""")

# Trigger AFTER INSERT
cursor.execute("""
CREATE TRIGGER auditoria_insert
AFTER INSERT ON productos
BEGIN
    INSERT INTO log_auditoria (tabla, operacion, registro_id, detalle)
    VALUES ('productos', 'INSERT', NEW.id,
            'Nuevo producto: ' || NEW.nombre || ' precio=' || NEW.precio);
END
""")

# Trigger AFTER UPDATE
cursor.execute("""
CREATE TRIGGER auditoria_update
AFTER UPDATE ON productos
BEGIN
    INSERT INTO log_auditoria (tabla, operacion, registro_id, detalle)
    VALUES ('productos', 'UPDATE', NEW.id,
            'Cambio: ' || OLD.nombre || '→' || NEW.nombre ||
            ' precio=' || OLD.precio || '→' || NEW.precio);
END
""")

# Trigger AFTER DELETE
cursor.execute("""
CREATE TRIGGER auditoria_delete
AFTER DELETE ON productos
BEGIN
    INSERT INTO log_auditoria (tabla, operacion, registro_id, detalle)
    VALUES ('productos', 'DELETE', OLD.id,
            'Eliminado: ' || OLD.nombre || ' precio=' || OLD.precio);
END
""")

# Operaciones que activan los triggers
cursor.execute("INSERT INTO productos (nombre, precio, stock) VALUES ('Notebook', 1200.0, 5)")
cursor.execute("INSERT INTO productos (nombre, precio, stock) VALUES ('Monitor', 350.0, 12)")
conn.commit()

cursor.execute("UPDATE productos SET precio = 1100.0 WHERE nombre = 'Notebook'")
conn.commit()

cursor.execute("DELETE FROM productos WHERE nombre = 'Monitor'")
conn.commit()

# Consultar el log de auditoría
print("=== Log de Auditoría Automático ===")
logs = cursor.execute("""
    SELECT operacion, tabla, registro_id, detalle
    FROM log_auditoria ORDER BY id
""").fetchall()

for log in logs:
    print(f"  [{log[0]}] tabla={log[1]}, id={log[2]}")
    print(f"    → {log[3]}")

print(f"\nTotal de eventos registrados: {len(logs)}")
conn.close()


## Vistas con JOIN y Agregación para Reportes

> Crea un conjunto de vistas que simplifican la generación de reportes de ventas: una vista detallada con JOINs y una vista de resumen con agregaciones. Demuestra cómo las vistas ocultan la complejidad de las consultas.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Esquema de ventas
cursor.execute("CREATE TABLE clientes (id INTEGER PRIMARY KEY, nombre TEXT, ciudad TEXT)")
cursor.execute("""
CREATE TABLE productos (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    categoria TEXT,
    precio REAL
)
""")
cursor.execute("""
CREATE TABLE ventas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    cliente_id INTEGER REFERENCES clientes(id),
    producto_id INTEGER REFERENCES productos(id),
    cantidad INTEGER,
    fecha TEXT DEFAULT (date('now'))
)
""")

# Datos de ejemplo
cursor.executemany("INSERT INTO clientes VALUES (?,?,?)", [
    (1, 'Empresa Alpha', 'Madrid'),
    (2, 'Empresa Beta', 'Barcelona'),
    (3, 'Empresa Gamma', 'Valencia')
])
cursor.executemany("INSERT INTO productos VALUES (?,?,?,?)", [
    (1, 'Laptop Pro', 'Electrónica', 1500.0),
    (2, 'Silla Ergonómica', 'Mobiliario', 450.0),
    (3, 'Monitor 4K', 'Electrónica', 800.0)
])
cursor.executemany("INSERT INTO ventas (cliente_id, producto_id, cantidad) VALUES (?,?,?)", [
    (1, 1, 3), (1, 3, 2), (2, 2, 5), (2, 1, 1), (3, 3, 4), (3, 2, 2)
])
conn.commit()

# Vista detallada de ventas
cursor.execute("""
CREATE VIEW ventas_detalle AS
SELECT v.id AS venta_id,
       c.nombre AS cliente,
       c.ciudad,
       p.nombre AS producto,
       p.categoria,
       v.cantidad,
       p.precio AS precio_unitario,
       ROUND(v.cantidad * p.precio, 2) AS subtotal
FROM ventas v
JOIN clientes c ON c.id = v.cliente_id
JOIN productos p ON p.id = v.producto_id
""")

# Vista de resumen por cliente
cursor.execute("""
CREATE VIEW resumen_por_cliente AS
SELECT cliente, ciudad,
       COUNT(*) AS num_compras,
       SUM(cantidad) AS unidades_totales,
       ROUND(SUM(subtotal), 2) AS total_gastado
FROM ventas_detalle
GROUP BY cliente, ciudad
ORDER BY total_gastado DESC
""")

# Vista de resumen por categoría
cursor.execute("""
CREATE VIEW resumen_por_categoria AS
SELECT categoria,
       COUNT(*) AS num_ventas,
       SUM(cantidad) AS unidades_vendidas,
       ROUND(SUM(subtotal), 2) AS ingresos_totales
FROM ventas_detalle
GROUP BY categoria
""")

print("=== Detalle de Ventas ===")
for row in cursor.execute("SELECT * FROM ventas_detalle"):
    print(f"  Venta #${row[0]}: ${row[1]} compró ${row[5]}x ${row[3]} = $${row[7]}")

print("\n=== Resumen por Cliente ===")
for row in cursor.execute("SELECT * FROM resumen_por_cliente"):
    print(f"  ${row[0]} (${row[1]}): ${row[2]} compras, total=$${row[4]}")

print("\n=== Resumen por Categoría ===")
for row in cursor.execute("SELECT * FROM resumen_por_categoria"):
    print(f"  ${row[0]}: ${row[1]} ventas, ${row[2]} unidades, ingresos=$${row[3]}")

conn.close()


## Funciones Python como Stored Procedures
Implementa un módulo de gestión de inventario usando funciones Python que actúan como stored procedures: encapsulan lógica compleja, manejan transacciones y retornan resultados estructurados.


In [ ]:
import sqlite3

# ── Módulo de Procedimientos de Inventario ──────────────────────────────────

def inicializar_bd(conn):
    c = conn.cursor()
    c.executescript("""
    CREATE TABLE IF NOT EXISTS almacenes (
        id INTEGER PRIMARY KEY,
        nombre TEXT,
        ubicacion TEXT
    );
    CREATE TABLE IF NOT EXISTS inventario (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        almacen_id INTEGER REFERENCES almacenes(id),
        producto TEXT,
        cantidad INTEGER DEFAULT 0,
        minimo INTEGER DEFAULT 5,
        UNIQUE(almacen_id, producto)
    );
    CREATE TABLE IF NOT EXISTS transferencias (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        origen_id INTEGER, destino_id INTEGER,
        producto TEXT, cantidad INTEGER,
        fecha TEXT DEFAULT (datetime('now'))
    );
    """)
    conn.commit()

def registrar_entrada(conn, almacen_id, producto, cantidad):
    """SP: Registrar entrada de stock."""
    c = conn.cursor()
    c.execute("""
    INSERT INTO inventario (almacen_id, producto, cantidad)
    VALUES (?, ?, ?)
    ON CONFLICT(almacen_id, producto)
    DO UPDATE SET cantidad = cantidad + excluded.cantidad
    """, (almacen_id, producto, cantidad))
    conn.commit()
    nuevo = c.execute(
        "SELECT cantidad FROM inventario WHERE almacen_id=? AND producto=?",
        (almacen_id, producto)
    ).fetchone()[0]
    print(f"  Entrada: +{cantidad} {producto} → stock total: {nuevo}")

def transferir_stock(conn, origen_id, destino_id, producto, cantidad):
    """SP: Transferir stock entre almacenes con validación."""
    c = conn.cursor()
    disponible = c.execute(
        "SELECT cantidad FROM inventario WHERE almacen_id=? AND producto=?",
        (origen_id, producto)
    ).fetchone()
    if not disponible or disponible[0] < cantidad:
        print(f"  ERROR: Stock insuficiente de '{producto}' en almacén {origen_id}")
        return False
    try:
        c.execute("UPDATE inventario SET cantidad = cantidad - ? WHERE almacen_id=? AND producto=?",
                  (cantidad, origen_id, producto))
        c.execute("""
        INSERT INTO inventario (almacen_id, producto, cantidad)
        VALUES (?, ?, ?)
        ON CONFLICT(almacen_id, producto)
        DO UPDATE SET cantidad = cantidad + excluded.cantidad
        """, (destino_id, producto, cantidad))
        c.execute("INSERT INTO transferencias (origen_id, destino_id, producto, cantidad) VALUES (?,?,?,?)",
                  (origen_id, destino_id, producto, cantidad))
        conn.commit()
        print(f"  Transferencia OK: {cantidad}x {producto} de almacén {origen_id} → {destino_id}")
        return True
    except Exception as e:
        conn.rollback()
        print(f"  Error en transferencia: {e}")
        return False

def reporte_stock_bajo(conn):
    """SP: Reporte de productos bajo el mínimo."""
    c = conn.cursor()
    rows = c.execute("""
    SELECT a.nombre, i.producto, i.cantidad, i.minimo
    FROM inventario i JOIN almacenes a ON a.id = i.almacen_id
    WHERE i.cantidad < i.minimo
    ORDER BY i.cantidad ASC
    """).fetchall()
    return rows

# ── Ejecución ────────────────────────────────────────────────────────────────
conn = sqlite3.connect(':memory:')
inicializar_bd(conn)

c = conn.cursor()
c.executemany("INSERT INTO almacenes VALUES (?,?,?)", [
    (1, 'Almacén Central', 'Madrid'),
    (2, 'Almacén Norte', 'Bilbao')
])
conn.commit()

print("=== Entradas de Stock ===")
registrar_entrada(conn, 1, 'Teclado', 50)
registrar_entrada(conn, 1, 'Mouse', 30)
registrar_entrada(conn, 2, 'Teclado', 3)

print("\n=== Transferencias ===")
transferir_stock(conn, 1, 2, 'Teclado', 10)
transferir_stock(conn, 1, 2, 'Mouse', 200)

print("\n=== Productos con Stock Bajo ===")
alertas = reporte_stock_bajo(conn)
if alertas:
    for row in alertas:
        print(f"  ⚠ $${row[0]}: $${row[1]} tiene $${row[2]} unidades (mínimo: $${row[3]})")
else:
    print("  Todo el inventario está sobre el mínimo.")

conn.close()


## Tips y Mejores Prácticas

> Los triggers en SQLite se ejecutan por operación de fila, no por sentencia. Si un UPDATE afecta 100 filas, el trigger se dispara 100 veces. Para operaciones masivas esto puede ser costoso en rendimiento. Considera desactivar temporalmente los triggers con PRAGMA recursive_triggers = OFF o hacer los cambios masivos en una sola transacción.

> Usa RAISE(ABORT, 'mensaje') dentro de un trigger BEFORE para cancelar la operación con un mensaje de error descriptivo. RAISE(IGNORE) simplemente cancela sin error. RAISE(FAIL, 'msg') cancela la sentencia pero no revierte la transacción completa. Elige según la severidad de la validación.

> Las vistas en SQLite no son materializadas: no almacenan datos propios. Cada consulta a una vista ejecuta su SELECT subyacente en tiempo real. Para consultas muy frecuentes sobre vistas complejas con JOINs, considera crear una tabla con índices adecuados o usar CTEs (WITH ... AS) dentro de la consulta en lugar de una vista.

> El patrón de funciones Python como "stored procedures" tiene una ventaja sobre los SP reales: puedes usar toda la lógica de Python (condicionales complejos, acceso a APIs, manejo de archivos) dentro del procedimiento. Agrupa las funciones en una clase o módulo dedicado para mantener el código organizado y testeable.


## Errores Comunes

### Trigger recursivo infinito

¿Por qué ocurre?
- Si un trigger AFTER UPDATE sobre la tabla A ejecuta otro UPDATE sobre la tabla A, puede activar el mismo trigger de nuevo, creando un bucle infinito que bloquea la base de datos.

Solución
- Usa una condición de guarda en el trigger (por ejemplo, verificar que el valor realmente cambió: WHEN NEW.valor <> OLD.valor). También puedes usar PRAGMA recursive_triggers = OFF para evitar que un trigger se llame a sí mismo recursivamente.


### Consultar una vista con GROUP BY esperando poder hacer UPDATE

¿Por qué ocurre?
- Las vistas que contienen GROUP BY, DISTINCT, funciones de agregación (SUM, COUNT, AVG) o JOINs son de solo lectura en SQLite. Intentar INSERT/UPDATE sobre ellas genera un error "cannot modify view".

Solución
- Para hacer escribible una vista compleja, implementa un INSTEAD OF trigger que traduzca las operaciones INSERT/UPDATE/DELETE sobre la vista a operaciones sobre las tablas reales subyacentes. Alternativamente, opera directamente sobre las tablas base.


### No usar transacciones en funciones Python que emulan stored procedures

¿Por qué ocurre?
- Si una función ejecuta múltiples sentencias SQL sin transacción explícita y falla en la mitad, la base de datos queda en estado inconsistente. Por ejemplo, en una transferencia de dinero: se debita la cuenta origen pero no se acredita el destino.

Solución
- Siempre envuelve operaciones multi-paso en try/except con conn.commit() al final y conn.rollback() en el except. Esto garantiza atomicidad: o todo el procedimiento tiene éxito o ningún cambio persiste.


### Usar NEW u OLD fuera del contexto correcto del trigger

¿Por qué ocurre?
- NEW solo está disponible en triggers INSERT y UPDATE. OLD solo está disponible en triggers UPDATE y DELETE. Usar NEW en un trigger DELETE o OLD en un trigger INSERT genera un error en tiempo de ejecución.

Solución
- Recuerda: INSERT tiene solo NEW, DELETE tiene solo OLD, y UPDATE tiene ambos NEW y OLD. Diseña la lógica del trigger respetando qué referencias están disponibles según el tipo de evento que lo activa.